In [4]:
import sqlite3
import pandas as pd

# Create an in-memory database (wipes when the kernel restarts -- perfect for teaching)
conn = sqlite3.connect(':memory:')

# Helper: run a query and return a clean DataFrame
def q(sql):
    return pd.read_sql_query(sql, conn)

# Helper: run DDL / DML statements (CREATE, INSERT, UPDATE, DELETE)
def run(sql):
    conn.executescript(sql)
    conn.commit()

print('SQL engine ready.')
print('Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.')

SQL engine ready.
Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.


In [5]:
run("""
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(50),
    salary DECIMAL(10, 2),
    hire_date VARCHAR(20),
    department VARCHAR(50)
);

INSERT INTO employees VALUES
(1, 'Amy West', 60000.00, '2021-01-15', 'HR'),
(2, 'Ivy Lee', 75000.50, '2020-05-22', 'Sales'),
(3, 'joe smith', 80000.75, '2019-08-10', 'Marketing'),
(4, 'John White', 90000.00, '2020-11-05', 'Finance'),
(5, 'Jane Hill', 55000.25, '2022-02-28', 'IT'),
(6, 'Dave West', 72000.00, '2020-03-12', 'Marketing'),
(7, 'Fanny Lee', 85000.50, '2018-06-25', 'Sales'),
(8, 'Amy Smith', 95000.25, '2019-11-30', 'Finance'),
(9, 'Ivy Hill', 62000.75, '2021-07-18', 'IT'),
(10, 'Joe White', 78000.00, '2022-04-05', 'Marketing'),
(11, 'John Lee', 68000.50, '2018-12-10', 'HR'),
(12, 'Jane West', 89000.25, '2017-09-15', 'Sales'),
(13, 'Dave Smith', 60000.75, '2022-01-08', NULL),
(14, 'Fanny White', 72000.00, '2019-04-22', 'IT'),
(15, 'Amy Hill', 84000.50, '2020-08-17', 'Marketing'),
(16, 'Ivy West', 92000.25, '2021-02-03', 'Finance'),
(17, 'Joe Lee', 58000.75, '2018-05-28', 'IT'),
(18, 'John Smith', 77000.00, '2019-10-10', 'HR'),
(19, 'Jane Hill', 81000.50, '2022-03-15', 'Sales'),
(20, 'Dave White', 70000.25, '2017-12-20', 'Marketing');
""")


run("UPDATE employees SET department = 'Unassigned' WHERE department IS NULL;")


run("""
DELETE FROM employees
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM employees
    GROUP BY employee_name, hire_date
);
""")


conn.create_function("TITLE", 1, lambda x: x.title() if x else x)
run("UPDATE employees SET employee_name = TITLE(employee_name);")


run("UPDATE employees SET hire_date = TRIM(hire_date);")


stats = q("SELECT AVG(salary) as avg_s, COUNT(*) as count_s FROM employees")
avg_salary = stats['avg_s'][0]


outliers = q(f"SELECT * FROM employees WHERE ABS(salary - {avg_salary}) > 15000")
print("Potentiels Outliers (Ecart > 15k de la moyenne) :")
print(outliers)


run("UPDATE employees SET department = UPPER(department);")

print("\nTable nettoyée et normalisée :")
print(q("SELECT * FROM employees"))

Potentiels Outliers (Ecart > 15k de la moyenne) :
   employee_id employee_name    salary   hire_date  department
0            1      Amy West  60000.00  2021-01-15          HR
1            5     Jane Hill  55000.25  2022-02-28          IT
2            8     Amy Smith  95000.25  2019-11-30     Finance
3           13    Dave Smith  60000.75  2022-01-08  Unassigned
4           16      Ivy West  92000.25  2021-02-03     Finance
5           17       Joe Lee  58000.75  2018-05-28          IT

Table nettoyée et normalisée :
    employee_id employee_name    salary   hire_date  department
0             1      Amy West  60000.00  2021-01-15          HR
1             2       Ivy Lee  75000.50  2020-05-22       SALES
2             3     Joe Smith  80000.75  2019-08-10   MARKETING
3             4    John White  90000.00  2020-11-05     FINANCE
4             5     Jane Hill  55000.25  2022-02-28          IT
5             6     Dave West  72000.00  2020-03-12   MARKETING
6             7     Fanny Lee